In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from typing import List, Dict, Any

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain.chat_models import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.document_loaders import TextLoader, PyPDFLoader, DirectoryLoader
from langchain.schema import Document
from langchain.prompts import PromptTemplate

from dotenv import load_dotenv
import gradio as gr
import json
from datetime import datetime

print("✅ All libraries imported successfully!")
print(f"Current working directory: {os.getcwd()}")

In [ ]:
os.environ["OPENAI_API_KEY"] = ""

CONFIG = {
    "model_name": "gpt-3.5-turbo",
    "temperature": 0.1,
    "chunk_size": 1000,
    "chunk_overlap": 200,
    "max_retrieval_docs": 4,
    "embedding_model": "text-embedding-ada-002"
}

print("✅ Configuration setup complete!")
print(f"Model: {CONFIG['model_name']}")
print(f"Chunk size: {CONFIG['chunk_size']}")

In [ ]:
os.makedirs("faq_documents", exist_ok=True)


faq_content = {
    "company_info.txt": """
Company FAQ - TechCorp Solutions

Q: What is TechCorp Solutions?
A: TechCorp Solutions is a leading technology company specializing in AI-driven software solutions, cloud computing, and digital transformation services. Founded in 2020, we serve clients across various industries including healthcare, finance, and retail.

Q: Where is TechCorp Solutions located?
A: Our headquarters is located in San Francisco, California, with additional offices in New York, London, and Tokyo. We also have a fully remote workforce across 15 countries.

Q: What services does TechCorp Solutions offer?
A: We offer a comprehensive range of services including:
- Custom software development
- AI and machine learning solutions
- Cloud migration and management
- Data analytics and business intelligence
- Cybersecurity consulting
- Digital transformation strategy

Q: How can I contact TechCorp Solutions?
A: You can reach us through:
- Email: contact@techcorp.com
- Phone: +1-555-TECH-CORP
- Website: www.techcorp.com
- Live chat on our website (available 24/7)
""",

    "products.txt": """
Product Information - TechCorp Solutions

Q: What is CloudSync Pro?
A: CloudSync Pro is our flagship cloud management platform that allows businesses to seamlessly migrate, monitor, and optimize their cloud infrastructure. It supports AWS, Azure, and Google Cloud platforms with advanced automation features.

Q: How much does CloudSync Pro cost?
A: CloudSync Pro pricing starts at $99/month for small businesses (up to 50 resources), $299/month for medium businesses (up to 500 resources), and custom enterprise pricing for larger organizations.

Q: What is AI Assistant Plus?
A: AI Assistant Plus is our advanced AI-powered virtual assistant that can be integrated into websites, mobile apps, and customer service platforms. It uses natural language processing to understand and respond to customer queries.

Q: Does AI Assistant Plus support multiple languages?
A: Yes, AI Assistant Plus supports over 25 languages including English, Spanish, French, German, Chinese, Japanese, and Portuguese.

Q: What is the difference between Basic and Pro plans?
A: Basic plan includes core features and email support, while Pro plan includes advanced analytics, priority support, custom integrations, and dedicated account management.
""",

    "support.txt": """
Support and Technical FAQ - TechCorp Solutions

Q: What are your support hours?
A: Our support team is available:
- Basic Support: Monday-Friday, 9 AM - 6 PM PST
- Premium Support: 24/7 including weekends and holidays
- Emergency Support: 24/7 for critical issues (Enterprise customers only)

Q: How do I report a bug or technical issue?
A: You can report technical issues through:
- Support portal: support.techcorp.com
- Email: support@techcorp.com
- Phone: +1-555-SUPPORT
- In-app support chat (for product users)

Q: What is your SLA for response times?
A: Our Service Level Agreement guarantees:
- Critical issues: 1 hour response time
- High priority: 4 hours response time
- Medium priority: 24 hours response time
- Low priority: 48 hours response time

Q: Do you offer training for your products?
A: Yes, we offer comprehensive training programs:
- Online self-paced courses
- Live webinar sessions
- On-site training (for enterprise customers)
- Certification programs
- Documentation and video tutorials

Q: How do I upgrade my subscription?
A: You can upgrade your subscription by:
- Logging into your account dashboard
- Contacting your account manager
- Calling our sales team at +1-555-SALES
- Using the upgrade option in the product interface
"""
}

for filename, content in faq_content.items():
    with open(f"faq_documents/{filename}", "w", encoding="utf-8") as f:
        f.write(content)

print("✅ Sample FAQ documents created!")
print(f"Created {len(faq_content)} FAQ documents:")
for filename in faq_content.keys():
    print(f"  - {filename}")

In [ ]:
class FAQChatbot:
    def __init__(self, config):
        self.config = config
        self.embeddings = None
        self.vectorstore = None
        self.qa_chain = None
        self.documents = []
        
    def load_documents(self, directory_path):
        """Load documents from directory"""
        try:
            
            loader = DirectoryLoader(
                directory_path,
                glob="*.txt",
                loader_cls=TextLoader,
                loader_kwargs={"encoding": "utf-8"}
            )
            documents = loader.load()
            
            
            for doc in documents:
                doc.metadata['source'] = os.path.basename(doc.metadata['source'])
                doc.metadata['timestamp'] = datetime.now().isoformat()
            
            self.documents = documents
            print(f"✅ Loaded {len(documents)} documents")
            
    
            for i, doc in enumerate(documents):
                print(f"Document {i+1}: {doc.metadata['source']} ({len(doc.page_content)} characters)")
                
            return documents
            
        except Exception as e:
            print(f"❌ Error loading documents: {str(e)}")
            return []
    
    def split_documents(self, documents):
        """Split documents into chunks"""
        try:
            text_splitter = RecursiveCharacterTextSplitter(
                chunk_size=self.config['chunk_size'],
                chunk_overlap=self.config['chunk_overlap'],
                length_function=len,
                separators=["\n\n", "\n", "Q:", "A:", " ", ""]
            )
            
            chunks = text_splitter.split_documents(documents)
            
            print(f"✅ Split documents into {len(chunks)} chunks")
            
    
            for i, chunk in enumerate(chunks[:3]):  # Show first 3 chunks
                print(f"\nChunk {i+1} (from {chunk.metadata['source']}):")
                print(f"Content preview: {chunk.page_content[:200]}...")
                print(f"Length: {len(chunk.page_content)} characters")
                
            return chunks
            
        except Exception as e:
            print(f"❌ Error splitting documents: {str(e)}")
            return []

chatbot = FAQChatbot(CONFIG)


documents = chatbot.load_documents("faq_documents")
chunks = chatbot.split_documents(documents)

print(f"\n📊 Processing Summary:")
print(f"Total documents: {len(documents)}")
print(f"Total chunks: {len(chunks)}")
print(f"Average chunk size: {np.mean([len(chunk.page_content) for chunk in chunks]):.0f} characters")

In [ ]:
def create_vector_store(chunks, config):
    """Create vector store from document chunks"""
    try:
        print("🔄 Creating embeddings...")
        
        embeddings = OpenAIEmbeddings(
            model=config['embedding_model'],
            openai_api_key=os.environ.get("OPENAI_API_KEY")
        )
        

        print("🔄 Creating vector store...")
        vectorstore = Chroma.from_documents(
            documents=chunks,
            embedding=embeddings,
            persist_directory="./chroma_db",
            collection_name="faq_collection"
        )
        
    
        vectorstore.persist()
        
        print("✅ Vector store created successfully!")
        print(f"Vector store contains {vectorstore._collection.count()} embeddings")
        
        return embeddings, vectorstore
        
    except Exception as e:
        print(f"❌ Error creating vector store: {str(e)}")
        return None, None


embeddings, vectorstore = create_vector_store(chunks, CONFIG)


if vectorstore:
    print("\n🔍 Testing vector store with sample query...")
    test_query = "What services does TechCorp offer?"
    relevant_docs = vectorstore.similarity_search(test_query, k=2)
    
    print(f"Found {len(relevant_docs)} relevant documents for query: '{test_query}'")
    for i, doc in enumerate(relevant_docs):
        print(f"\nRelevant Document {i+1}:")
        print(f"Source: {doc.metadata['source']}")
        print(f"Content: {doc.page_content[:300]}...")

In [ ]:

def create_qa_chain(vectorstore, embeddings, config):
    """Create QA chain with custom prompt"""
    try:
        
        custom_prompt = PromptTemplate(
            template="""You are a helpful FAQ assistant for TechCorp Solutions. Use the following pieces of context to answer the user's question. If you don't know the answer based on the context, say that you don't have that information in the knowledge base.

Context: {context}

Question: {question}

Instructions:
- Provide accurate, helpful answers based on the context
- If the information is not in the context, say so politely
- Keep answers concise but complete
- Use a friendly, professional tone
- If relevant, mention related information that might be helpful

Answer:""",
            input_variables=["context", "question"]
        )
        
        
        llm = ChatOpenAI(
            model_name=config['model_name'],
            temperature=config['temperature'],
            openai_api_key=os.environ.get("OPENAI_API_KEY")
        )
        

        qa_chain = RetrievalQA.from_chain_type(
            llm=llm,
            chain_type="stuff",
            retriever=vectorstore.as_retriever(
                search_type="similarity",
                search_kwargs={"k": config['max_retrieval_docs']}
            ),
            chain_type_kwargs={"prompt": custom_prompt},
            return_source_documents=True
        )
        
        print("✅ QA chain created successfully!")
        return qa_chain
        
    except Exception as e:
        print(f"❌ Error creating QA chain: {str(e)}")
        return None


qa_chain = create_qa_chain(vectorstore, embeddings, CONFIG)


if qa_chain:
    print("\n🤖 Testing QA chain...")
    test_questions = [
        "What is TechCorp Solutions?",
        "How much does CloudSync Pro cost?",
        "What are your support hours?",
        "Do you offer training?"
    ]
    
    for question in test_questions:
        print(f"\n❓ Question: {question}")
        try:
            result = qa_chain({"query": question})
            print(f"🤖 Answer: {result['result']}")
            print(f"📄 Sources: {[doc.metadata['source'] for doc in result['source_documents']]}")
        except Exception as e:
            print(f"❌ Error: {str(e)}")

In [ ]:
class EnhancedFAQChatbot:
    def __init__(self, config):
        self.config = config
        self.embeddings = embeddings
        self.vectorstore = vectorstore
        self.qa_chain = qa_chain
        self.chat_history = []
        
    def get_answer(self, question, include_sources=True):
        """Get answer for a question"""
        try:
        
            result = self.qa_chain({"query": question})
            
            answer = result['result']
            sources = result['source_documents']
            
            response = {
                'question': question,
                'answer': answer,
                'sources': [doc.metadata['source'] for doc in sources] if include_sources else [],
                'confidence': self._calculate_confidence(sources),
                'timestamp': datetime.now().isoformat()
            }
            
            self.chat_history.append(response)
            
            return response
            
        except Exception as e:
            return {
                'question': question,
                'answer': f"I'm sorry, I encountered an error: {str(e)}",
                'sources': [],
                'confidence': 0.0,
                'timestamp': datetime.now().isoformat()
            }
    
    def _calculate_confidence(self, sources):
        """Calculate confidence score based on source documents"""
        if not sources:
            return 0.0
       
        base_confidence = min(len(sources) * 0.25, 1.0)
        return round(base_confidence, 2)
    
    def get_chat_history(self):
        """Get chat history"""
        return self.chat_history
    
    def clear_history(self):
        """Clear chat history"""
        self.chat_history = []
        
    def add_documents(self, new_documents):
        """Add new documents to the knowledge base"""
        try:
            
            text_splitter = RecursiveCharacterTextSplitter(
                chunk_size=self.config['chunk_size'],
                chunk_overlap=self.config['chunk_overlap']
            )
            
            new_chunks = text_splitter.split_documents(new_documents)
            
        
            self.vectorstore.add_documents(new_chunks)
            self.vectorstore.persist()
            
            return f"Added {len(new_chunks)} new chunks to the knowledge base"
            
        except Exception as e:
            return f"Error adding documents: {str(e)}"

enhanced_chatbot = EnhancedFAQChatbot(CONFIG)

print("✅ Enhanced FAQ Chatbot initialized!")
print("🚀 Ready to answer questions!")

In [ ]:
def interactive_chat():
    """Interactive chat function for testing"""
    print("\n🤖 FAQ Chatbot Interactive Mode")
    print("Type 'quit' to exit, 'history' to see chat history, 'clear' to clear history")
    print("-" * 50)
    
    while True:
        try:
            question = input("\n💬 Your question: ").strip()
            
            if question.lower() == 'quit':
                print("👋 Thank you for using the FAQ Chatbot!")
                break
                
            elif question.lower() == 'history':
                history = enhanced_chatbot.get_chat_history()
                print(f"\n📋 Chat History ({len(history)} messages):")
                for i, msg in enumerate(history, 1):
                    print(f"{i}. Q: {msg['question']}")
                    print(f"   A: {msg['answer'][:100]}...")
                continue
                
            elif question.lower() == 'clear':
                enhanced_chatbot.clear_history()
                print("🧹 Chat history cleared!")
                continue
                
            elif not question:
                print("Please enter a question.")
                continue
            
            
            print("🔄 Processing your question...")
            response = enhanced_chatbot.get_answer(question)
            
            
            print(f"\n🤖 Answer: {response['answer']}")
            print(f"📄 Sources: {', '.join(response['sources'])}")
            print(f"🎯 Confidence: {response['confidence']}")
            
        except KeyboardInterrupt:
            print("\n\n👋 Goodbye!")
            break
        except Exception as e:
            print(f"❌ Error: {str(e)}")


test_questions = [
    "What is TechCorp Solutions?",
    "How much does CloudSync Pro cost?",
    "What are your support hours?",
    "Do you offer training programs?",
    "How can I contact support?",
    "What languages does AI Assistant Plus support?",
    "What is the difference between Basic and Pro plans?"
]

print("🧪 Testing with predefined questions...")
for i, question in enumerate(test_questions, 1):
    print(f"\n{'='*50}")
    print(f"Test {i}: {question}")
    print('='*50)
    
    response = enhanced_chatbot.get_answer(question)
    print(f"🤖 Answer: {response['answer']}")
    print(f"📄 Sources: {', '.join(response['sources'])}")
    print(f"🎯 Confidence: {response['confidence']}")



In [ ]:

def create_gradio_interface():
    """Create Gradio web interface for the chatbot"""
    
    def chat_interface(message, history):
        """Chat interface function for Gradio"""
        if not message.strip():
            return history, history
         
        response = enhanced_chatbot.get_answer(message)

        bot_response = f"{response['answer']}\n\n"
        if response['sources']:
            bot_response += f"📄 **Sources:** {', '.join(response['sources'])}\n"
        bot_response += f"🎯 **Confidence:** {response['confidence']}"
        

        history.append([message, bot_response])
        
        return history, history
    
    def clear_chat():
        """Clear chat history"""
        enhanced_chatbot.clear_history()
        return [], []
    
  
    with gr.Blocks(title="FAQ Chatbot", theme=gr.themes.Soft()) as demo:
        gr.Markdown("# 🤖 TechCorp Solutions FAQ Chatbot")
        gr.Markdown("Ask me anything about TechCorp Solutions services, products, or support!")
        
        with gr.Row():
            with gr.Column(scale=4):
                chatbot_ui = gr.Chatbot(
                    value=[],
                    height=400,
                    show_label=False,
                    container=True
                )
                
                with gr.Row():
                    msg = gr.Textbox(
                        placeholder="Type your question here...",
                        show_label=False,
                        scale=4
                    )
                    send_btn = gr.Button("Send", variant="primary", scale=1)
                    clear_btn = gr.Button("Clear", variant="secondary", scale=1)
            
            with gr.Column(scale=1):
                gr.Markdown("### 💡 Sample Questions")
                sample_questions = [
                    "What is TechCorp Solutions?",
                    "How much does CloudSync Pro cost?",
                    "What are your support hours?",
                    "Do you offer training?",
                    "How can I contact support?"
                ]
                
                for question in sample_questions:
                    gr.Button(question, size="sm").click(
                        lambda q=question: (q, ""),
                        outputs=[msg, gr.Textbox()]
                    )
        
      
        msg.submit(chat_interface, [msg, chatbot_ui], [chatbot_ui, chatbot_ui])
        msg.submit(lambda: "", None, [msg])
        
        send_btn.click(chat_interface, [msg, chatbot_ui], [chatbot_ui, chatbot_ui])
        send_btn.click(lambda: "", None, [msg])
        
        clear_btn.click(clear_chat, None, [chatbot_ui, chatbot_ui])
        
        gr.Markdown("### 📊 Chatbot Statistics")
        stats_display = gr.Markdown("Ready to answer questions!")
        
        def update_stats():
            history = enhanced_chatbot.get_chat_history()
            stats = f"""
            - **Total Questions:** {len(history)}
            - **Average Confidence:** {np.mean([h['confidence'] for h in history]) if history else 0:.2f}
            - **Most Recent:** {history[-1]['timestamp'][:19] if history else 'N/A'}
            """
            return stats
        
      
        demo.load(update_stats, None, stats_display, every=5)
    
    return demo


print("🌐 Creating Gradio web interface...")
demo = create_gradio_interface()

print("🚀 Launching web interface...")
print("The interface will open in your browser automatically.")
print("If it doesn't open, click on the URL that appears below.")
print("✅ Gradio interface created successfully!")
print("To launch the interface, uncomment the demo.launch() line above.")

In [ ]:

class ChatbotAnalytics:
    def __init__(self, chatbot):
        self.chatbot = chatbot
        
    def evaluate_performance(self, test_questions_and_answers):
        """Evaluate chatbot performance"""
        results = []
        
        for question, expected_keywords in test_questions_and_answers.items():
            response = self.chatbot.get_answer(question)
            
            
            answer_lower = response['answer'].lower()
            keywords_found = [kw for kw in expected_keywords if kw.lower() in answer_lower]
            
            score = len(keywords_found) / len(expected_keywords) if expected_keywords else 0
            
            results.append({
                'question': question,
                'answer': response['answer'],
                'expected_keywords': expected_keywords,
                'keywords_found': keywords_found,
                'score': score,
                'confidence': response['confidence'],
                'sources': response['sources']
            })
            
        return results
    
    def generate_report(self, results):
        """Generate performance report"""
        if not results:
            return "No results to analyze."
        
        total_score = sum(r['score'] for r in results)
        avg_score = total_score / len(results)
        avg_confidence = sum(r['confidence'] for r in results) / len(results)
        
        report = f"""
📊 CHATBOT PERFORMANCE REPORT
{'='*50}

📈 Overall Metrics:
- Total Questions: {len(results)}
- Average Accuracy Score: {avg_score:.2f}/1.0 ({avg_score*100:.1f}%)
- Average Confidence: {avg_confidence:.2f}/1.0 ({avg_confidence*100:.1f}%)

📝 Detailed Results:
"""
        
        for i, result in enumerate(results, 1):
            report += f"""
{i}. Question: {result['question']}
   Score: {result['score']:.2f}/1.0
   Confidence: {result['confidence']:.2f}/1.0
   Keywords Found: {len(result['keywords_found'])}/{len(result['expected_keywords'])}
   Sources: {', '.join(result['sources'])}
   
"""
        
        return report


test_data = {
    "What is TechCorp Solutions?": ["technology", "company", "software", "AI"],
    "How much does CloudSync Pro cost?": ["$99", "pricing", "month", "small"],
    "What are your support hours?": ["Monday", "Friday", "9 AM", "6 PM"],
    "Do you offer training?": ["training", "courses", "webinar", "certification"],
    "How can I contact support?": ["support", "email", "phone", "chat"]
}


analytics = ChatbotAnalytics(enhanced_chatbot)
print("🔍 Evaluating chatbot performance...")

results = analytics.evaluate_performance(test_data)
report = analytics.generate_report(results)

print(report)


with open("chatbot_performance_report.txt", "w") as f:
    f.write(report)

print("💾 Performance report saved to 'chatbot_performance_report.txt'")

In [ ]:

def export_chatbot_data():
    """Export chatbot data for deployment"""
    try:
        
        config_data = {
            'config': CONFIG,
            'model_info': {
                'embedding_model': CONFIG['embedding_model'],
                'llm_model': CONFIG['model_name'],
                'vector_store_path': './chroma_db',
                'document_count': len(enhanced_chatbot.vectorstore._collection.get()['documents'])
            },
            'export_timestamp': datetime.now().isoformat()
        }
        
        with open('chatbot_config.json', 'w') as f:
            json.dump(config_data, f, indent=2)
        
        
        history = enhanced_chatbot.get_chat_history()
        with open('chat_history.json', 'w') as f:
            json.dump(history, f, indent=2)
        
        
        deployment_guide = """
# FAQ Chatbot Deployment Guide

## Files Included:
- `chatbot_config.json`: Configuration settings
- `chat_history.json`: Chat history data
- `chroma_db/`: Vector database directory
- `faq_documents/`: Source documents

## Deployment Steps:

### 1. Environment Setup
```bash
pip install langchain openai chromadb sentence-transformers tiktoken gradio python-dotenv

In [ ]:
export OPENAI_API_KEY="your-api-key-here"

In [ ]:
from langchain.vectorstores import Chroma
from langchain.embeddings import OpenAIEmbeddings
from langchain.chat_models import ChatOpenAI
from langchain.chains import RetrievalQA


with open('chatbot_config.json', 'r') as f:
    config = json.load(f)


embeddings = OpenAIEmbeddings()


vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings
)

llm = ChatOpenAI(model_name=config['config']['model_name'])
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever()
)